mix them pixels yooo

In [151]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier 
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.multioutput import MultiOutputRegressor, RegressorChain
import linear_mixing as lmx
import scipy.stats as stats
from scipy.optimize import curve_fit

In [2]:
# Magic function to auto-update imported first-party scripts 
%load_ext autoreload
%autoreload 2

In [416]:
# Plotting settings

plt.rcParams['font.family'] = 'serif' #sans-serif'

# mixing pixels

In [ ]:
material_options = {
    # "kelp" : ['ecklonia', 'macrocystis', 'undaria'],
    # "brown_non_kelp" : ['acrocarpia', 'cystophora', 'carpophyllum', 
    #                 'durvillaea', 'hormosira', 'petalonia','phyllospora', 'sargassum', 'scytosiphon'],
    "brown_algae" : ['ecklonia', 'macrocystis', 'petalonia', 
                     'undaria','acrocarpia', 'cystophora', 'carpophyllum', 'durvillaea', 'hormosira', 'phyllospora', 'sargassum', 'scytosiphon'],
    "red_veg" : ['rhodophyte', 'filamentous_rhodophyte','frondose_rhodophyte',],
    "green_veg" : ['grass', 'ulva'],
    # "non_brown_autos" : ['grass', 'filamentous_rhodophyte',
    #                      'frondose_rhodophyte', 'rhodophyte', 'ulva'],
    # "all_non_kelp_autos" : ['acrocarpia', 'carpophyllum', 'cystophora', 
    #                         'durvillaea', 'filamentous_rhodophyte','frondose_rhodophyte', 'petalonia', 'grass','hormosira','phyllospora', 'rhodophyte',  'sargassum','scytosiphon','ulva'],
    # "all_non_kelp" : ['acrocarpia', 'barnacle_shells', 'carpophyllum',
    #                   'cystophora', 'durvillaea', 'filamentous_rhodophyte','frondose_rhodophyte', 'gravel', 'grass','hormosira','mussels','petalonia','phyllospora', 'rhodophyte', 'rock','sand', 'sargassum', 'scytosiphon', 'shell_litter','rock', 'ulva', 'worm_castings'],
    "mineral" : ['barnacle_shells', 'gravel', 'mussels', 'rock', 'sand', 
                  'shell_litter', 'worm_castings'],
    }
material_options.keys()

In [267]:
# #Assign fractional cover ranges with simulate within
# cover_ranges = {
#     "kelp" : (0,0.8),
#     #"brown_algae" : (0, 0.5), 
#     "brown_non_kelp" : (0,0.8),
#     "red_veg" : (0,0.2),
#     "green_veg" : (0, 0.5),
#     #"non_brown_autos" : (0, 40),
#     #"all_non_kelp_autos" : (0, 100),
#     #"all_non_kelp" : (0,100),
#     "mineral" : (0,0.7),}

cover_ranges = {
    # "kelp" : (0,1),
    # "brown_non_kelp" : (0,1),
    "brown_algae" : (0, 1),
    "red_veg" : (0,1),
    "green_veg" : (0, 1),
    #"non_brown_autos" : (0, 40),
    #"all_non_kelp_autos" : (0, 100),
    #"all_non_kelp" : (0,100),
    "mineral" : (0,1),}

In [269]:
data = pd.read_csv("./data/processed/resampled/noisy_Sentinel_2_ABC_resampled.csv", index_col=0)
labels = pd.read_csv("data/labels_prepped.csv", index_col=0)
data = pd.concat([labels["Class"], data], axis = 1)

In [ ]:
waters = pd.read_csv("./S2_deep_water_agg.csv", index_col=0)
waters = waters.iloc[:, 4:-3]
print(f"total of {waters.shape[0]} water pixels loaded")
waters.drop_duplicates(inplace = True)
print(f"Total of {waters.shape[0]} unique water pixels kept")
waters= waters/10000
waters.head()

In [270]:
mixer = lmx.LinearMixing(materials = material_options, cover_ranges = cover_ranges, pixels = 4000, material_spectra= data, water_spectra = waters)


In [271]:
results, f = mixer.sim_many_pixels()

In [ ]:
simulated_pixels, pixel_fpcs = mixer.format_sim_results(results, f)
del results, f
simulated_pixels.head()

In [ ]:
pixel_fpcs.head()

In [ ]:
# Plot the components of the simulated pixels

args = {'log' : True} 

fig, ax = plt.subplots(3,2)
ax[0,0].hist(pixel_fpcs['kelp'], bins = 50,range = (0.00000000000, max(pixel_fpcs['kelp'])), **args) #log = True)
# ax[0,1].hist(pixel_fpcs['brown_non_kelp'], bins = 50, range = (0.00000000000, max(pixel_fpcs['brown_non_kelp'])), **args)
# ax[1,0].hist(pixel_fpcs['red_veg'], bins = 50, range = (0.00000000000, max(pixel_fpcs['red_veg'])), **args)
# ax[1,1].hist(pixel_fpcs['green_veg'], bins = 50, range = (0.00000000000, max(pixel_fpcs['green_veg'])), **args)
ax[2,0].hist(pixel_fpcs['mineral'], bins = 50, range = (0.00000000000, max(pixel_fpcs['mineral'])), **args)
ax[2,1].hist(pixel_fpcs['water'], bins = 50)
plt.show()

# fig.savefig(r"C:\Users\s4770224\Documents\Work\Writing\Figures\Obj1\part2\simulated_pixel_fpcs_histograms.svg")
# plt.show()

In [ ]:
plt.hist(pixel_fpcs['non-water_comps'])
# fig.savefig(r"C:\Users\s4770224\Documents\Work\Writing\Figures\Obj1\part2\simulated_pixel_component_counts.svg")
plt.show()

In [21]:
#simulated_pixels.to_csv("data/mixed_sims/kbrgm_simulated_pixels_18112025.csv")
# pixel_fpcs.to_csv("data/mixed_sims/kbrgm_pixel_fpcs_18112025.csv")

# Prepare data for classification

In [276]:
# Define what you simulated
label_level = "kbrgm"
keep_list = ["kelp", "water", "other_brown_alg", "mineral", "green_veg", "red_veg"]

# Add water to classifier training set 
sample_idx = np.random.choice(waters.index, size = 1000, replace = False)
water_labels = pd.DataFrame("water", index = sample_idx, columns = labels.columns)
water_sample = waters.iloc[sample_idx, :]
water_sample.columns = data.columns[1:]
data_watered = pd.concat([data, water_sample], axis = 0)
labels_watered = pd.concat([labels, water_labels], axis = 0).set_index(data_watered.index)
data_watered = pd.concat([labels_watered, data_watered.iloc[:, 1:]], axis = 1)


In [278]:
labels_watered["brgm"] = labels_watered["kbrgm"].replace(to_replace=["kelp", "brown_non_kelp"], value = "browns")

In [280]:
# Filter training data to only mixed pixel components
data_watered = data_watered[data_watered[label_level].isin(keep_list)]
water_endmember = data_watered[label_level] == "water"


In [281]:
# Extract PCA components and transform dataset
deco = PCA(8)
deco.fit(data_watered.iloc[:,8:])
decomposed_endmembers = deco.transform(data_watered.iloc[:, 8:])
decomposed_mixpix = deco.transform(simulated_pixels)
comps = deco.components_

In [ ]:
plt.plot(comps, label = range(comps.shape[1]))
plt.legend()
plt.show()

In [283]:
pixel_fpcs["maj"] = pixel_fpcs.iloc[:, :-1].idxmax(axis = 1)
fpcs = pixel_fpcs.iloc[:, :-2]

In [311]:
# divide the dataset into training and testing
x_train, x_test, y_train, y_test = train_test_split(decomposed_mixpix, fpcs, train_size = 0.5, random_state = 4)
x_train = pd.DataFrame(x_train)
x_test = pd.DataFrame(x_test)

# Perform Multi output regression

In [381]:
multi = MultiOutputRegressor(RandomForestRegressor(n_estimators = 100, criterion = "squared_error", random_state=4, oob_score= True))
multi.fit(x_train, y_train)
l = multi.predict(x_test)

In [ ]:
# Calc R squared
r2_l = 1 - (np.sum((y_test - l)**2, axis = 0) / np.sum((y_test - np.mean(y_test, axis = 0))**2, axis = 0))
print(f"R squared values for each component:\n{r2_l}")

In [ ]:
for mat in range(y_test.shape[1]):
    plt.scatter(y_test.iloc[:, mat], l.iloc[:,mat], alpha = 0.4, c = l.iloc[:,4], cmap = "viridis")
    plt.xlabel(f"fractional cover of {fpcs.columns[mat]}")
    plt.ylabel ("Predicted cover")
    plt.annotate(f"R squared = {r2_l[mat]:.2f}", xy = (0.7, 0.1), xycoords = "axes fraction")
    plt.show()

In [ ]:
l_sum = np.sum(l, axis = 1)
plt.hist(l_sum, bins = 50)
plt.show()

In [ ]:
# Adjust to sum to 1
l_adj = l/ l_sum[:, None]

# Calc R squared on adjusted values
r2_l_adj = 1 - (np.sum((y_test - l_adj)**2, axis = 0) / np.sum((y_test - np.mean(y_test, axis = 0))**2, axis = 0))
print(f"R squared values for each adjusted component:\n{r2_l_adj}")

for mat in range(y_test.shape[1]):
    plt.scatter(y_test.iloc[:, mat], l_adj[:,mat], alpha = np.sqrt(l_adj[:,4]), color = "lightseagreen") # c = l_adj[:,4]
    plt.xlabel(f"fractional cover of {fpcs.columns[mat]}")
    plt.xlim(0,1)
    plt.ylabel ("Predicted cover")
    plt.ylim(0,1)
    plt.annotate(f"R squared = {r2_l_adj[mat]:.3f}", xy = (0.7, 0.1), xycoords = "axes fraction")
    plt.axline((0,0), slope = 1, color = "black", linestyle = "--")
    plt.show()

In [ ]:
# Calc R squared

r2_l_adj = 1 - (np.sum((y_test - l_adj)**2, axis = 0) / np.sum((y_test - np.mean(y_test, axis = 0))**2, axis = 0))
print(f"R squared values for each component:\n{r2_l_adj}")

In [ ]:
for mat in range(y_test.shape[1]):
    plt.scatter(y_test.iloc[:, mat], l_adj.iloc[:,mat], alpha = 0.4, c = l_adj.iloc[:,4], cmap = "viridis")
    plt.xlabel(f"fractional cover of {fpcs.columns[mat]}")
    plt.ylabel ("Predicted cover")
    plt.annotate(f"R squared = {r2_l[mat]:.2f}", xy = (0.7, 0.1), xycoords = "axes fraction")
    plt.show()

# Perform Chained regression

In [312]:
rfreg = RandomForestRegressor(n_estimators = 100, criterion = "squared_error", oob_score = True)
chain = RegressorChain(rfreg, order = [4, 0, 3, 1, 2]).fit(x_train, y_train)
p = chain.predict(x_test)

In [ ]:
# Calc R squared

r2 = 1 - (np.sum((y_test - p)**2, axis = 0) / np.sum((y_test - np.mean(y_test, axis = 0))**2, axis = 0))
print(f"R squared values for each component:\n{r2}")

In [ ]:
for mat in range(y_test.shape[1]):
    plt.scatter(y_test.iloc[:, mat], p[:,mat], alpha = 0.4, c = p[:,4], cmap = "viridis")
    plt.xlabel(f"fractional cover of {fpcs.columns[mat]}")
    plt.ylabel ("Predicted cover")
    plt.annotate(f"R squared = {r2[mat]:.2f}", xy = (0.7, 0.1), xycoords = "axes fraction")
    plt.show()

In [ ]:
p_sum = np.sum(p, axis = 1)
plt.hist(p_sum, bins = 50)
plt.show()

In [ ]:
# Adjust to sum to 1
p_adj = p/ p_sum[:, None]

# Calc R squared on adjusted values
r2_adj = 1 - (np.sum((y_test - p_adj)**2, axis = 0) / np.sum((y_test - np.mean(y_test, axis = 0))**2, axis = 0))
print(f"R squared values for each adjusted component:\n{r2_adj}")

for mat in range(y_test.shape[1]):
    plt.scatter(y_test.iloc[:, mat], p_adj[:,mat], alpha = 0.4, c = p_adj[:,4], cmap = "viridis")
    plt.xlabel(f"fractional cover of {fpcs.columns[mat]}")
    plt.ylabel ("Predicted cover")
    plt.annotate(f"R squared = {r2_adj[mat]:.3f}", xy = (0.7, 0.1), xycoords = "axes fraction")
    plt.show()

# Perform Soft classification

In [248]:
labels = data_watered.iloc[:, :8]
decomposed_endmembers

soft_c = RandomForestClassifier(n_estimators = 100, criterion = "gini", random_state=4, oob_score= True)
soft_c.fit(decomposed_endmembers, labels["kbrgm"])
pred = soft_c.predict_proba(decomposed_mixpix)

In [ ]:
labels["kbrgm"].unique()

In [ ]:
for mat in range(y_test.shape[1]):
    plt.scatter(fpcs.iloc[:, mat], pred[:,mat], alpha = 0.2)
    plt.xlabel(f"fractional cover of {fpcs.columns[mat]}")
    plt.ylabel ("Predicted cover")
    plt.show()